In [8]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Rosmaster_Lib import Rosmaster
from ipywidgets import interact
import ipywidgets as widgets
import struct

In [ ]:

# Create the Rosmaster object bot
bot = Rosmaster()
bot._Rosmaster__debug = True  # Correct way to access private attribute
# Start to receive data, can only start once, all read data function is based on this method
bot.create_receive_threading()

def set_pid_param(self, kp, ki, kd, motor_index=0, forever=False):
    try:
        state = 0
        if forever:
            state = 0x5F
        
        # motor_index: 0 = all motors, 1-4 = individual motors
        cmd = [self._Rosmaster__HEAD, self._Rosmaster__DEVICE_ID, 0x0A, self.FUNC_SET_MOTOR_PID, int(motor_index) & 0xff]
        
        if kp > 10 or ki > 10 or kd > 10 or kp < 0 or ki < 0 or kd < 0:
            print("PID value must be:[0, 10.00]")
            return
        
        kp_params = bytearray(struct.pack('h', int(kp * 1000)))
        ki_params = bytearray(struct.pack('h', int(ki * 1000)))
        kd_params = bytearray(struct.pack('h', int(kd * 1000)))
        
        cmd.append(kp_params[0])  # low
        cmd.append(kp_params[1])  # high
        cmd.append(ki_params[0])  # low
        cmd.append(ki_params[1])  # high
        cmd.append(kd_params[0])  # low
        cmd.append(kd_params[1])  # high
        cmd.append(state)
        
        checksum = sum(cmd, self._Rosmaster__COMPLEMENT) & 0xff
        cmd.append(checksum)
        self.ser.write(cmd)
        
        if self._Rosmaster__debug:
            print("pid:", cmd)
        time.sleep(self._Rosmaster__delay_time)
        if forever:
            time.sleep(.1)
    except:
        print('---set_pid_param error!---')
        pass

# Bind the new method to the instance using types.MethodType
import types
bot.set_pid_param = types.MethodType(set_pid_param, bot)

Rosmaster Serial Opened! Baudrate=115200
----------------create receive threading--------------
check sum error: 19 13 [152, 0, 0, 0, 199, 255, 255, 255, 255, 255, 255, 255, 252, 255, 255, 113, 251]
check sum error: 21 14 [0, 5, 220, 245, 189, 94, 104, 228, 237, 78, 255, 251, 12, 160, 255, 0, 0, 39, 251]


Exception in thread task_serial_receive:
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/dist-packages/Rosmaster_Lib-3.3.9-py3.11.egg/Rosmaster_Lib/Rosmaster_Lib.py", line 260, in __receive_data
  File "/usr/lib/python3/dist-packages/serial/serialposix.py", line 595, in read
    raise SerialException(
serial.serialutil.SerialException: device reports readiness to read but returned no data (device disconnected or multiple access on port?)


In [10]:
kp = 0.8
ki = 0.5
kd = 0.6
bot.set_pid_param(bot, kp=kp, ki=ki, kd=kd,motor_index=1, forever=False)

TypeError: set_pid_param() got multiple values for argument 'kp'

In [11]:
bot.set_pid_param(6, 0.01, 0.7, motor_index=1)

pid: [255, 252, 10, 19, 1, 112, 23, 10, 0, 188, 2, 0, 109]


In [12]:
# Now you can use it with motor_index parameter
# Set PID for all motors (original behavior)
bot.set_pid_param(1.0, 0.1, 0.05, motor_index=0)

# Set PID for motor 1 only
bot.set_pid_param(1.2, 0.15, 0.06, motor_index=1)

# Set PID for motor 2 only  
bot.set_pid_param(0.8, 0.08, 0.04, motor_index=2)

pid: [255, 252, 10, 19, 0, 232, 3, 100, 0, 50, 0, 0, 158]
pid: [255, 252, 10, 19, 1, 176, 4, 150, 0, 60, 0, 0, 164]
pid: [255, 252, 10, 19, 2, 32, 3, 80, 0, 40, 0, 0, 186]


In [13]:
bot.set_car_run(1,20,adjust=True);
time.sleep(5);
bot.set_car_run(3,20,adjust=True);
time.sleep(2);
bot.set_car_run(4,20,adjust=True);
time.sleep(2);
bot.set_car_run(2,20,adjust=True);
time.sleep(5);
bot.set_car_run(0,0,adjust=True);

car_run: [255, 252, 7, 17, 129, 1, 20, 0, 174]
car_run: [255, 252, 7, 17, 129, 3, 20, 0, 176]
car_run: [255, 252, 7, 17, 129, 4, 20, 0, 177]
car_run: [255, 252, 7, 17, 129, 2, 20, 0, 175]
car_run: [255, 252, 7, 17, 129, 0, 0, 0, 153]


Exception in thread task_serial_receive:
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/dist-packages/Rosmaster_Lib-3.3.9-py3.11.egg/Rosmaster_Lib/Rosmaster_Lib.py", line 260, in __receive_data
  File "/usr/lib/python3/dist-packages/serial/serialposix.py", line 595, in read
    raise SerialException(
serial.serialutil.SerialException: device reports readiness to read but returned no data (device disconnected or multiple access on port?)


FUNC_SET_MOTOR_PID: 1 [1000, 80, 100]
check sum error: 10 0 [0, 0, 121, 255, 21, 0, 8, 2]
check sum error: 21 14 [251, 255, 0, 0, 2, 0, 116, 0, 0, 113, 217, 139, 251, 12, 255, 0, 1, 13, 255]
